# 🤖 RAG Chatbot — MILOUDI — NLP 2025
**GitHub:** https://github.com/Melouka-mld/RAG-Chatbot-NLP_MILOUDI
**Tutorial:** LangChain + RAG (Streamlit/Gradio)



---
## 📌 RAG Pipeline
```
data/comments.txt
   ↓ TextLoader
chunks (800 chars, overlap 80)
   ↓ all-MiniLM-L6-v2
vectors (384 dims) → FAISS database
   ↓
User question → embed → FAISS search → top-3 chunks → LLM → Answer
```


---
## 📦 CELL 1 —  numpy + Install


In [1]:
import subprocess, sys

print('Step 1/3: Fixing numpy conflict...')
subprocess.run([sys.executable,'-m','pip','uninstall','-y',
    'numpy','scipy','scikit-learn','sentence-transformers'],
    capture_output=True)
subprocess.run([sys.executable,'-m','pip','install','-q',
    'numpy==1.26.4'],capture_output=True)
print('  ✓ numpy==1.26.4')

print('Step 2/3: Installing ML packages...')
for p in ['sentence-transformers','faiss-cpu','scipy','scikit-learn']:
    subprocess.run([sys.executable,'-m','pip','install','-q',p],capture_output=True)
    print(f'  ✓ {p}')

print('Step 3/3: Installing LangChain + UI...')
for p in ['langchain==0.2.16','langchain-community==0.2.16',
          'langchain-core==0.2.38','huggingface_hub',
          'transformers','pypdf','python-dotenv','requests','gradio']:
    subprocess.run([sys.executable,'-m','pip','install','-q',p],capture_output=True)
    print(f'  ✓ {p}')

print('\n ALL INSTALLED!')


Step 1/3: Fixing numpy conflict...
  ✓ numpy==1.26.4
Step 2/3: Installing ML packages...
  ✓ sentence-transformers
  ✓ faiss-cpu
  ✓ scipy
  ✓ scikit-learn
Step 3/3: Installing LangChain + UI...
  ✓ langchain==0.2.16
  ✓ langchain-community==0.2.16
  ✓ langchain-core==0.2.38
  ✓ huggingface_hub
  ✓ transformers
  ✓ pypdf
  ✓ python-dotenv
  ✓ requests
  ✓ gradio

 ALL INSTALLED!


---
##  CELL 2 — Clone repo + HuggingFace token


In [1]:
import os, subprocess
from getpass import getpass

REPO = 'https://github.com/Melouka-mld/RAG-Chatbot-NLP_MILOUDI.git'
NAME = 'RAG-Chatbot-NLP_MILOUDI'

# Remove old clone
if os.path.exists(f'/content/{NAME}'):
    subprocess.run(f'rm -rf /content/{NAME}', shell=True)
    print('Removed old copy')

r = subprocess.run(f'git clone {REPO}', shell=True,
                   capture_output=True, text=True)
if r.returncode == 0:
    os.chdir(f'/content/{NAME}')
    print(f'✅ Cloned! Folder: {os.getcwd()}')
    files = [f for f in os.listdir('.') if not f.startswith('.')]
    print(f'Files: {files}')
else:
    print('❌ Clone failed:', r.stderr[:200])
    print('Fix: Go to GitHub → Settings → Change visibility → Make Public')

# Token
HF_TOKEN = getpass(' Paste HuggingFace token (hf_...): ')

if HF_TOKEN.startswith('hf_'):
    os.environ['HUGGINGFACEHUB_API_TOKEN'] = HF_TOKEN
    with open('.env','w') as f:
        f.write(f'HUGGINGFACEHUB_API_TOKEN={HF_TOKEN}\n')
    print(f'✅ Token set: {HF_TOKEN[:10]}...')
else:
    print('⚠️  Token must start with hf_ — try again')

✅ Cloned! Folder: /content/RAG-Chatbot-NLP_MILOUDI
Files: ['chatbot_rag.py', 'data', 'retrieval.py', 'requirements.txt', 'README.md', 'ingestion.py']
 Paste HuggingFace token (hf_...): ··········
✅ Token set: hf_RpvWCSy...


---
## 🏗️ CELL 3 — Build FAISS database
Loads text → splits into chunks → embeds → saves FAISS index

⏳ First time downloads embedding model (~80MB) — wait 3-4 min

In [2]:
import os, sys
sys.path.insert(0,'.')

from langchain_community.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Find data file
for candidate in ['data/comments.txt','data/sample_comments.txt']:
    if os.path.exists(candidate):
        DATA = candidate
        break

print(f' Loading: {DATA}')
loader = TextLoader(DATA, encoding='utf-8')
documents = loader.load()
print(f'   ✅ {len(documents)} document loaded')
print(f'   Size: {len(documents[0].page_content)} chars')

# Chunk
print('\n  Splitting into chunks...')
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800, chunk_overlap=80)
chunks = splitter.split_documents(documents)
print(f'   ✅ {len(chunks)} chunks (800 chars each, 80 overlap)')
print(f'   Sample chunk: {chunks[0].page_content[:100]}...')

# Embed
print('\n Loading embedding model (first time ~3 min)...')
emb = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2',
    model_kwargs={'device':'cpu'},
    encode_kwargs={'normalize_embeddings':True}
)
print('   ✅ Model loaded: all-MiniLM-L6-v2 (384-dim vectors)')

# Store
print('\n Building FAISS index...')
db = FAISS.from_documents(chunks, emb)
db.save_local('faiss_db')

print(f'\n✅ FAISS DATABASE READY!')
print(f'   Vectors: {db.index.ntotal}')
print(f'   Files:   {os.listdir("faiss_db")}')

 Loading: data/comments.txt
   ✅ 1 document loaded
   Size: 4444 chars

  Splitting into chunks...
   ✅ 7 chunks (800 chars each, 80 overlap)
   Sample chunk: CUSTOMER COMMENTS DATASET — E-COMMERCE PLATFORM
Dat...

 Loading embedding model (first time ~3 min)...


/tmp/ipykernel_16411/1431591794.py:31: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  emb = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

   ✅ Model loaded: all-MiniLM-L6-v2 (384-dim vectors)

 Building FAISS index...

✅ FAISS DATABASE READY!
   Vectors: 7
   Files:   ['index.pkl', 'index.faiss']


---
## 🔍 CELL 4 — Test retrieval (no LLM needed)
This proves RAG search works. FAISS finds relevant chunks by meaning.

In [3]:
print('='*55)
print('RETRIEVAL TEST — semantic search (no LLM)')
print('='*55)

test_qs = [
    'What do customers say about delivery?',
    'What are the main complaints?',
    'How is customer service rated?'
]

for q in test_qs:
    print(f'\n❓ {q}')
    results = db.similarity_search(q, k=2)
    for i,doc in enumerate(results,1):
        print(f'  [{i}] {doc.page_content[:150].strip()}...')

print('\n✅ RETRIEVAL WORKS!')
print('   FAISS finds relevant chunks by semantic similarity.')
print('   Next: pass these chunks to the LLM for a full answer.')

RETRIEVAL TEST — semantic search (no LLM)

❓ What do customers say about delivery?
  [1] Most Common Complaints:
- Delayed shipping beyond promised dates
- Wrong or broken items received
- Poor customer service response time
- Product qual...
  [2] Service Comment 4 (Negative):
Sent 4 emails over 2 weeks with no reply whatsoever.
The phone line always goes straight to voicemail.
Completely unacce...

❓ What are the main complaints?
  [1] Most Common Complaints:
- Delayed shipping beyond promised dates
- Wrong or broken items received
- Poor customer service response time
- Product qual...
  [2] Service Comment 4 (Negative):
Sent 4 emails over 2 weeks with no reply whatsoever.
The phone line always goes straight to voicemail.
Completely unacce...

❓ How is customer service rated?
  [1] Service Comment 4 (Negative):
Sent 4 emails over 2 weeks with no reply whatsoever.
The phone line always goes straight to voicemail.
Completely unacce...
  [2] CUSTOMER COMMENTS DATASET — E-COMMERCE PLATF

---
##  CELL 5 — Full RAG + LLM answers
Tries multiple free HuggingFace models.


In [4]:
import os, time, requests

HF_TOKEN = os.environ.get('HUGGINGFACEHUB_API_TOKEN','')

PROMPT = """Use ONLY the context below to answer in 1-2 sentences.
If the answer is not there say: I do not have enough information.

Context:
{context}

Question: {question}
Answer:"""

# Models to try (flan-t5 removed from free API in 2025)
MODELS = [
    'mistralai/Mistral-7B-Instruct-v0.3',
    'HuggingFaceH4/zephyr-7b-beta',
    'microsoft/Phi-3-mini-4k-instruct',
    'tiiuae/falcon-7b-instruct',
    'google/gemma-2-2b-it',
]

def call_api(prompt, model):
    url = f'https://api-inference.huggingface.co/models/{model}'
    try:
        r = requests.post(
            url,
            headers={'Authorization':f'Bearer {HF_TOKEN}'},
            json={'inputs':prompt[:900],
                  'parameters':{'max_new_tokens':120,
                                'do_sample':False,
                                'return_full_text':False},
                  'options':{'wait_for_model':True}},
            timeout=45
        )
        if r.status_code==200 and r.text.strip():
            res = r.json()
            if isinstance(res,list) and res:
                t = res[0].get('generated_text','')
                if t.strip(): return t.strip()
    except: pass
    return None

# Find working model
WORKING_MODEL = None
print('🔧 Testing models...')
for m in MODELS:
    short = m.split('/')[-1]
    ans = call_api('Complete this: 2+2=', m)
    status = '✅' if ans else '❌'
    print(f'  {status} {short}')
    if ans:
        WORKING_MODEL = m
        print(f'  → Using: {m}')
        break
    time.sleep(1)

if not WORKING_MODEL:
    print('\n⚠️  No API model available — RAG retrieval mode (still valid!)')
else:
    print(f'\n✅ LLM ready!')

# Ask 3 questions
def ask(question):
    print(f'\n❓ {question}')
    docs = db.similarity_search(question, k=3)
    ctx  = '\n\n'.join(d.page_content[:250] for d in docs)

    if WORKING_MODEL:
        prompt = PROMPT.format(context=ctx, question=question)
        answer = call_api(prompt, WORKING_MODEL)
        answer = answer or docs[0].page_content[:200]
    else:
        answer = docs[0].page_content[:200] + '...'

    print(f' {answer}')
    print(f' {len(docs)} chunks used')
    return answer

ask('What do customers say about delivery speed?')
time.sleep(2)
ask('What are the most common complaints?')
time.sleep(2)
ask('How long did the refund process take?')

🔧 Testing models...
  ❌ Mistral-7B-Instruct-v0.3
  ❌ zephyr-7b-beta
  ❌ Phi-3-mini-4k-instruct
  ❌ falcon-7b-instruct
  ❌ gemma-2-2b-it

⚠️  No API model available — RAG retrieval mode (still valid!)

❓ What do customers say about delivery speed?
 Most Common Complaints:
- Delayed shipping beyond promised dates
- Wrong or broken items received
- Poor customer service response time
- Product quality not matching the price
- Refund process taking...
 3 chunks used

❓ What are the most common complaints?
 Most Common Complaints:
- Delayed shipping beyond promised dates
- Wrong or broken items received
- Poor customer service response time
- Product quality not matching the price
- Refund process taking...
 3 chunks used

❓ How long did the refund process take?
 === CUSTOMER SERVICE FEEDBACK ===

Service Comment 1 (Positive):
The customer service team was absolutely excellent!
They resolved my issue in under 10 minutes. Friendly, professional, and efficient.
...
 3 chunks used


'=== CUSTOMER SERVICE FEEDBACK ===\n\nService Comment 1 (Positive):\nThe customer service team was absolutely excellent!\nThey resolved my issue in under 10 minutes. Friendly, professional, and efficient.\n...'

---
## 📊 CELL 6 — Evaluation table


In [5]:
import pandas as pd, time

eval_questions = [
    'What do customers say about delivery speed?',
    'What are the most common complaints?',
    'How satisfied are customers with product quality?',
    'How was the customer service described?',
    'How long did the refund process take?',
]

results = []
print(' Evaluation (5 questions)\n')

for i,q in enumerate(eval_questions):
    print(f'[{i+1}/5] {q}')
    t0   = time.time()
    docs = db.similarity_search(q, k=3)
    ctx  = '\n\n'.join(d.page_content[:250] for d in docs)

    if WORKING_MODEL:
        prompt = PROMPT.format(context=ctx, question=q)
        answer = call_api(prompt, WORKING_MODEL) or docs[0].page_content[:200]
        time.sleep(2)
    else:
        answer = docs[0].page_content[:200] + '...'

    results.append({
        'Question': q,
        'Answer'  : answer[:150],
        'Method'  : 'LLM+RAG' if WORKING_MODEL else 'RAG retrieval',
        'Time(s)' : round(time.time()-t0,1),
        'Chunks'  : len(docs)
    })
    print(f'   ✓ {answer[:80]}')

df = pd.DataFrame(results)
pd.set_option('display.max_colwidth', 80)
print(f'\n✅ EVALUATION COMPLETE')
print(f'Method: {"LLM+RAG" if WORKING_MODEL else "RAG retrieval only"}')
print(f'Avg time: {df["Time(s)"].mean():.1f}s\n')
display(df)
df.to_csv('evaluation_results.csv', index=False)
print('\n Saved: evaluation_results.csv')


 Evaluation (5 questions)

[1/5] What do customers say about delivery speed?
   ✓ Most Common Complaints:
- Delayed shipping beyond promised dates
- Wrong or brok
[2/5] What are the most common complaints?
   ✓ Most Common Complaints:
- Delayed shipping beyond promised dates
- Wrong or brok
[3/5] How satisfied are customers with product quality?
   ✓ Most Common Complaints:
- Delayed shipping beyond promised dates
- Wrong or brok
[4/5] How was the customer service described?
   ✓ Service Comment 4 (Negative):
Sent 4 emails over 2 weeks with no reply whatsoeve
[5/5] How long did the refund process take?
   ✓ === CUSTOMER SERVICE FEEDBACK ===

Service Comment 1 (Positive):
The customer se

✅ EVALUATION COMPLETE
Method: RAG retrieval only
Avg time: 0.0s



,Question,Answer,Method,Time(s),Chunks
0,What do customers say about delivery speed?,Most Common Complaints:\n- Delayed shipping beyond promised dates\n- Wrong o...,RAG retrieval,0.0,3
1,What are the most common complaints?,Most Common Complaints:\n- Delayed shipping beyond promised dates\n- Wrong o...,RAG retrieval,0.0,3
2,How satisfied are customers with product quality?,Most Common Complaints:\n- Delayed shipping beyond promised dates\n- Wrong o...,RAG retrieval,0.0,3
3,How was the customer service described?,Service Comment 4 (Negative):\nSent 4 emails over 2 weeks with no reply what...,RAG retrieval,0.0,3
4,How long did the refund process take?,=== CUSTOMER SERVICE FEEDBACK ===\n\nService Comment 1 (Positive):\nThe cust...,RAG retrieval,0.0,3



 Saved: evaluation_results.csv


---
##  CELL 7 — Launch Gradio chatbot


In [6]:
import gradio as gr, os, time

PROMPT_CHAT = """Answer using ONLY the context below in 1-2 sentences.
If not in context say: I do not have enough information.

Context:
{context}

Question: {question}
Answer:"""

def chat(message, history):
    if not message.strip(): return 'Please type a question.'
    docs = db.similarity_search(message, k=3)
    ctx  = '\n\n'.join(d.page_content[:250] for d in docs)
    src  = '\n'.join(f'• {d.page_content[:80]}...' for d in docs)

    answer = None
    if WORKING_MODEL:
        answer = call_api(
            PROMPT_CHAT.format(context=ctx, question=message),
            WORKING_MODEL
        )
    if not answer:
        answer = docs[0].page_content[:300]

    return f'{answer}\n\n---\n Retrieved chunks:\n{src}'

with gr.Blocks(
    title='RAG Chatbot — MILOUDI NLP 2025',
    theme=gr.themes.Soft(primary_hue='blue')
) as demo:
    gr.Markdown('#  RAG Chatbot — MILOUDI — NLP 2025')
    gr.Markdown('**GitHub:** Melouka-mld/RAG-Chatbot-NLP_MILOUDI | '
                '**DB:** FAISS | **Embeddings:** all-MiniLM-L6-v2')
    gr.Markdown('Ask questions about the **customer comments dataset**!')
    gr.ChatInterface(
        fn=chat,
        examples=[
            'What do customers say about delivery speed?',
            'What are the most common complaints?',
            'How satisfied are customers with product quality?',
            'How was the customer service described?',
            'How long did the refund take?',
        ],
        cache_examples=False
    )

print(' Launching chatbot — a public link will appear below...')
demo.launch(share=True)

/tmp/ipykernel_16411/1100737050.py:29: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


 Launching chatbot — a public link will appear below...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://dc7ff3c0610d50efa0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


---
##  CELL 8 — Push to GitHub


In [12]:
import subprocess
from getpass import getpass



TOKEN = getpass(' GitHub token: ')
EMAIL = 'meloukamiloudi@email.com'

def run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip()
    if out: print(out)
    return r.returncode

run('git config --global user.name "Melouka mld"')
run(f'git config --global user.email "{EMAIL}"')
run('git add .')

# Check if there is anything to commit
status = subprocess.run('git status --short', shell=True,
    capture_output=True, text=True).stdout.strip()
if status:
    run('git commit -m "Final verified RAG chatbot — MILOUDI NLP 2025"')
else:
    print('Nothing new to commit (already up to date)')

remote = f'https://{TOKEN}@github.com/Melouka-mld/RAG-Chatbot-NLP_MILOUDI.git'
run(f'git remote set-url origin {remote}')
code = run('git push origin main')

if code == 0:
    print('\n PUSHED SUCCESSFULLY!')
    print('🔗 https://github.com/Melouka-mld/RAG-Chatbot-NLP_MILOUDI')
else:
    print('\n❌ Push failed — generate a fresh token and try again')

 GitHub token: ··········
Nothing new to commit (already up to date)
To https://github.com/Melouka-mld/RAG-Chatbot-NLP_MILOUDI.git
   6570cd0..115a3a6  main -> main

 PUSHED SUCCESSFULLY!
🔗 https://github.com/Melouka-mld/RAG-Chatbot-NLP_MILOUDI
